<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>


# Lecture — Introduction to Pipeline Deployment

## Overview

This lecture covers pipeline (streaming) deployment — serving a model continuously as new data arrives, instead of on a fixed batch schedule.

This lecture covers 4 sections that build on each other:

- **A. Pipeline / Streaming Deployment Fundamentals**. What streaming deployment is, its advantages and limitations, and the typical streaming inference workflow.
- **B. Lakeflow Spark Declarative Pipelines**. SDP as the declarative way to do ETL and inference on Databricks, and its building blocks — streaming tables and materialized views.
- **C. Pipeline Features**. Built-in data quality expectations, pipeline observability, and continuous or scheduled data ingestion.
- **D. Building and Running a Pipeline**. The three steps to create your first Lakeflow pipeline and add streaming inference as the final step.

## Learning Objectives

By the end of this lecture, you will be able to:

1. Describe pipeline deployment and identify scenarios in which this method is required.
2. Describe Lakeflow Spark Declarative Pipelines as a tool to develop and manage inference pipelines.
3. Explain the pipeline features that support data quality, observability, and continuous ingestion.
4. Outline how to develop a simple Lakeflow Declarative Pipeline that performs streaming-based inference in its final step.

## A. Pipeline / Streaming Deployment Fundamentals

### A1. What Is Pipeline / Streaming Deployment

**Streaming deployment** means deploying machine learning models that continuously perform inference as data arrives, providing low-latency, real-time insights instead of processing data in batches.


<div style="max-width: 1000px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 75" role="img" style="font-family: sans-serif;">
  <title>Batch Inference</title>
  <defs>
    <marker id="arrowBatch" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto">
      <polygon points="0 0, 8 3.5, 0 7" fill="#1B3139"/>
    </marker>
  </defs>
  <text x="20" y="50" font-size="15" font-weight="bold" fill="#1565C0">Batch</text>
  <rect x="120" y="25" width="110" height="42" rx="6" fill="#E3F2FD" stroke="#1976D2" stroke-width="1.2"/>
  <text x="174" y="51" text-anchor="middle" font-size="13" fill="#1B3139">collect</text>
  <line x1="230" y1="45.4" x2="252" y2="45.4" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowBatch)"/>
  <rect x="252" y="25" width="110" height="42" rx="6" fill="#E3F2FD" stroke="#1976D2" stroke-width="1.2"/>
  <text x="306" y="51" text-anchor="middle" font-size="13" fill="#1B3139">wait</text>
  <line x1="362" y1="45.4" x2="384" y2="45.4" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowBatch)"/>
  <rect x="384" y="25" width="135" height="42" rx="6" fill="#1B3139"/>
  <text x="450" y="51" text-anchor="middle" font-size="13" fill="#FFFFFF">score batch</text>
  <text x="525" y="51" font-size="15" fill="#607D8B">&#8594; on a schedule</text>

<div style="max-width: 1000px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 100" role="img" style="font-family: sans-serif;">
  <title>Streaming Inference</title>
  <text x="20" y="60" font-size="15" font-weight="bold" fill="#E65100">Streaming</text>
  <rect x="125" y="35" width="400" height="42" rx="6" fill="#FFF3E0" stroke="#F57C00" stroke-width="1.2"/>
  <text x="330" y="60" text-anchor="middle" font-size="13" fill="#1B3139">score each record continuously as it arrives</text>
  <text x="535" y="60" font-size="15" fill="#E65100">&#8594; low latency, continuous</text>
</svg>
</div>

### A2. Advantages and Limitations

Streaming deployment buys fresher predictions than batch, at the cost of more complexity and compute. It sits between batch and true real-time serving.

</br>

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.green{--accent:#00A972;} .coral{--accent:#FF5F46;}
.ds-card-title { font-size: 16pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 14pt; color: #5E7077; line-height: 1.55; margin: 0; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.tc { text-align: center; }
.dd-head { font-size: 16pt; font-weight: 700; margin: 4px 0 12px; color: var(--accent); }
</style>

<div class="ds-row">
  <div class="ds-card green">
    <div class="dd-head">ADVANTAGES</div>
    <ul>
      <li><strong>Lower latency</strong> predictions</li>
      <li>Generate predictions and <strong>act sooner</strong></li>
      <li><strong>Event-driven</strong> architecture; enables systems to adapt quickly</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="dd-head">LIMITATIONS</div>
    <ul>
      <li>More <strong>costly</strong> than a batch solution</li>
      <li>More <strong>complicated</strong> to develop, maintain, and monitor</li>
      <li>Less throughput than batch</li>
      <li><strong>Resource intensiveness</strong> — computational power to process data</li>
    </ul>
  </div>
</div>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#1B5162; margin-bottom:6px; font-size:15pt;">Note</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    Streaming deployment provides faster predictions than batch, but is not intended for applications needing true sub-second (real-time) inference. For those, use a real-time serving endpoint.
  </div>
</div>

### A3. The Streaming Inference Workflow

The streaming workflow mirrors the batch workflow, with one key difference: instead of a scheduled job scoring a fixed batch, the inference step is **continuously triggered as new data arrives** from a Lakeflow SDP pipeline or other streaming sources.

<div style="max-width: 1000px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 880 320" role="img" style="font-family: sans-serif;">
  <title>A Typical Streaming Model Deployment Workflow</title>

  <!-- Build model -->
  <rect x="15" y="30" width="150" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="90" y="50" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Delta Lake /</text>
  <text x="90" y="68" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Feature Store</text>

  <rect x="15" y="120" width="150" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="90" y="142" text-anchor="middle" font-size="13" font-weight="bold" fill="#1B3139">Model Training</text>
  <text x="90" y="158" text-anchor="middle" font-size="11" fill="#1B3139">TF, scikit-learn, XGBoost</text>

  <rect x="205" y="120" width="150" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="280" y="142" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Models in</text>
  <text x="280" y="159" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Unity Catalog</text>

  <!-- Dashed continuous region -->
  <rect x="390" y="20" width="475" height="285" rx="12" fill="none" stroke="#7B1FA2" stroke-width="1.8" stroke-dasharray="7,5"/>
  <text x="627" y="42" text-anchor="middle" font-size="12" font-weight="bold" fill="#6A1B9A">Continuous triggering as new data arrives</text>

  <!-- Streaming sources -->
  <rect x="410" y="60" width="130" height="60" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="1.5"/>
  <text x="475" y="85" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Lakeflow SDP</text>
  <text x="475" y="107" text-anchor="middle" font-size="10" font-weight="bold" fill="#2E7D32">New Data</text>

  <rect x="560" y="60" width="130" height="60" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="1.5"/>
  <text x="625" y="82" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Other Streaming</text>
  <text x="625" y="97" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Sources</text>
  <text x="625" y="113" text-anchor="middle" font-size="10" font-weight="bold" fill="#2E7D32">New Data</text>

  <!-- Streaming inference -->
  <rect x="470" y="165" width="160" height="70" rx="8" fill="#FFF8E1" stroke="#F57C00" stroke-width="2"/>
  <text x="550" y="195" text-anchor="middle" font-size="13" font-weight="bold" fill="#E65100">Streaming Inference</text>
  <text x="550" y="217" text-anchor="middle" font-size="12" fill="#1B3139" font-family="monospace">pyfunc.predict</text>

  <!-- Downstream -->
  <rect x="690" y="140" width="160" height="40" rx="6" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.3"/>
  <text x="700" y="164" font-size="12" font-weight="bold" fill="#1B3139">Data Lake / Database</text>
  <rect x="690" y="190" width="160" height="40" rx="6" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.3"/>
  <text x="700" y="214" font-size="11" fill="#607D8B">Any downstream consumer</text>

  <!-- Arrows -->
  <line x1="90" y1="85" x2="90" y2="120" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowP"/>
  <line x1="90" y1="120" x2="90" y2="100" stroke="#1B3139" stroke-width="0" />
  <line x1="165" y1="147" x2="205" y2="147" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowP)"/>
  <line x1="355" y1="147" x2="470" y2="190" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowP)"/>
  <line x1="475" y1="120" x2="530" y2="165" stroke="#2E7D32" stroke-width="1.5" marker-end="url(#arrowPg)"/>
  <line x1="625" y1="120" x2="575" y2="165" stroke="#2E7D32" stroke-width="1.5" marker-end="url(#arrowPg)"/>
  <path d="M630 192 C660 192, 665 160, 690 160" stroke="#1B3139" stroke-width="1.5" fill="none" marker-end="url(#arrowP)"/>
  <path d="M630 200 C660 200, 665 210, 690 210" stroke="#1B3139" stroke-width="1.5" fill="none" marker-end="url(#arrowP)"/>

  <defs>
    <marker id="arrowP" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto"><polygon points="0 0, 8 3.5, 0 7" fill="#1B3139"/></marker>
    <marker id="arrowPg" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto"><polygon points="0 0, 8 3.5, 0 7" fill="#2E7D32"/></marker>
  </defs>
</svg>
</div>

## B. Lakeflow Spark Declarative Pipelines

### B1. SDP — The Declarative Way to Build Pipelines

**Lakeflow Spark Declarative Pipelines (SDP)** is the declarative way to do ETL and inference on the Databricks Data Intelligence Platform. You declare the tables you want in **SQL or Python**, and SDP automatically orchestrates the DAG, handles retries, and manages changing data.

</br>

<div style="max-width: 900px; margin: 0 auto; display: flex; gap: 24px; align-items: flex-start;">

<div style="flex: 1;">

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>CREATE STREAMING TABLE raw_data
AS SELECT * FROM STREAM
  read_files('/raw_data', format => 'json');
</br>
CREATE MATERIALIZED VIEW clean_data
AS SELECT ...
FROM raw_data</code>
</pre>
</div>

</div>

<div style="flex: 1;">
<ul style="margin: 0; padding-left: 18px; line-height: 1.7;">
  <li><strong>Accelerate ETL development</strong> — declare SQL or Python and SDP orchestrates the DAG, handles retries and changing data.</li>
  <li><strong>Automatically manage your infrastructure</strong> — recovery, auto-scaling, and performance optimization.</li>
  <li><strong>Ensure high data quality</strong> — built-in quality controls, testing, monitoring, and enforcement.</li>
  <li><strong>Unify batch and streaming</strong> — the simplicity of SQL with the freshness of streaming, in one unified API.</li>
</ul>
</div>

</div>

### B2. Streaming Tables and Materialized Views

**Streaming tables** and **materialized views** are the building blocks of a Lakeflow pipeline. Both are a kind of **dataset** — defined by a SQL query and created and kept up to date by a pipeline.

- A **streaming table** ingests and processes new records incrementally as they arrive — ideal for raw ingestion and streaming inference.
- A **materialized view** stores the precomputed result of a query and refreshes as its inputs change — ideal for transformations and aggregations.

<div style="max-width: 900px; margin: 0 auto; display: flex; gap: 24px; align-items: flex-start;">

<div style="flex: 1;">

**A Dataset is:**
- Defined by a **SQL query**
- Created and kept **up-to-date** by a pipeline

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>CREATE MATERIALIZED VIEW report
AS SELECT sum(profit)
FROM prod.sales</code>
</pre>
</div>

</div>

<div style="flex: 1; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px;">
<strong style="color: #0d47a1;">Lakeflow pipelines provide tools to:</strong>
<ul style="margin: 10px 0 0 0; padding-left: 20px; line-height: 1.9;">
  <li><strong>Manage</strong> dependencies</li>
  <li><strong>Control</strong> quality</li>
  <li><strong>Automate</strong> operations</li>
  <li><strong>Simplify</strong> collaboration</li>
  <li><strong>Save</strong> costs</li>
  <li><strong>Reduce</strong> latency</li>
</ul>
</div>

</div>

## C. Pipeline Features

### C1. Data Quality Validation and Monitoring

Lakeflow SDP lets you define **data quality and integrity controls** directly in the pipeline using **expectations**. Each expectation is a constraint with a policy for what to do when a row violates it: **warn** (retain the row), **drop** the row, or **fail** the pipeline. All pipeline runs and quality metrics are captured, tracked, and reported.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>-- Stage 1: Bronze table — drop invalid rows
CREATE OR REFRESH STREAMING TABLE fire_account_bronze (
  CONSTRAINT valid_account_open_dt
    EXPECT (
      account_open_dt IS NOT NULL
      AND account_close_dt > account_open_dt
    )
    ON VIOLATION DROP ROW
)
COMMENT "Bronze table with valid account IDs"
AS
SELECT * FROM STREAM(fire_account_raw);</code>
</pre>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;"><strong>The Three Policies:</strong>
    <ul style="margin: 8px 0 0 0; padding-left: 20px;">
      <li><strong>WARN</strong>: Keep the row and record the violation.</li>
      <li><strong>DROP ROW</strong>: Discard the invalid row.</li>
      <li><strong>FAIL UPDATE</strong>: Stop the pipeline.</li>
    </ul>
    The Data Quality dashboard reports records processed, written, and dropped per expectation.
  </div>
</div>

### C2. Data Pipeline Observability

Lakeflow SDP gives you visibility into how a pipeline runs and how data flows through it — essential for operating an inference pipeline in production.

<table style="border-collapse: collapse; width: 100%; max-width: 900px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; margin: 16px 0;">
  <thead>
    <tr style="background-color: #1B3139;">
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Capability</th>
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">What it provides</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Lineage diagram</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">High-quality, high-fidelity view of how data flows through the pipeline, for impact analysis.</td>
    </tr>
    <tr>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Granular logging</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Operational, governance, quality, and status logging down to the row level.</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Monitoring</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Continuously monitor pipeline jobs to ensure continued operation.</td>
    </tr>
    <tr>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Notifications</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Alerting on pipeline events using Databricks SQL.</td>
    </tr>
  </tbody>
</table>

### C3. Continuous or Scheduled Data Ingestion

A streaming inference pipeline needs a reliable way to pick up new data. Lakeflow SDP ingests files **incrementally and efficiently as they arrive** in cloud storage using **Auto Loader**, and it can run either continuously or on a schedule.

- **Incrementally** and efficiently process new data files as they arrive in cloud storage using **Auto Loader**.
- Automatically **infer schema** of incoming files, or superimpose what you know with **schema hints**.
- Automatic **schema evolution** — supported for JSON, CSV, Avro, and Parquet.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>-- Simple SQL syntax for streaming ingestion
CREATE STREAMING TABLE sales_orders_raw
COMMENT "The raw sales orders, ingested from cloud storage."
AS SELECT * FROM STREAM
  read_files('/databricks-datasets/retail-org/sales_orders/', format => 'json');</code>
</pre>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#1B5162; margin-bottom:6px; font-size:15pt;">Note</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    Auto Loader powers incremental file ingestion in Lakeflow SDP. The current SQL syntax uses <code>read_files(...)</code> inside a <code>STREAM</code> source; older examples may reference the legacy <code>cloud_files(...)</code> function.
  </div>
</div>

## D. Building and Running a Pipeline

### D1. Creating Your First Lakeflow Pipeline

Going from SQL to a running Lakeflow pipeline takes three steps.

</br>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 14pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 12pt; color: #5E7077; line-height: 1.55; margin: 0; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.tc { text-align: center; }
.ds-card-label { font-size: 14pt; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; color: var(--accent); margin-bottom: 4px; }
.ds-card-para { font-size: 14pt; color: #0b2026; line-height: 1.55; margin: 0; }
</style>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-label">Step One</div>
    <div class="ds-card-title">Write Definitions</div>
    <div class="ds-card-para">Write your streaming table / materialized view definitions (but do not run them) in notebooks or source files. Git folders let you version-control them.</div>
  </div>
  <div class="ds-card green">
    <div class="ds-card-label">Step Two</div>
    <div class="ds-card-title">Create a Pipeline</div>
    <div class="ds-card-para">A pipeline picks one or more source files of table definitions, plus any required configuration.</div>
  </div>
  <div class="ds-card amber">
    <div class="ds-card-label">Step Three</div>
    <div class="ds-card-title">Click Start</div>
    <div class="ds-card-para">Lakeflow SDP will create or update all the tables in the pipeline.</div>
  </div>
</div>
</div>

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>CREATE MATERIALIZED VIEW daily_stats
AS SELECT sum(rev) - sum(costs) AS profits
FROM prod_data.transactions
GROUP BY day</code>
</pre>
</div>

### D2. Streaming Inference as the Final Step

To turn an ETL pipeline into an **inference pipeline**, make the final dataset a streaming table (or materialized view) whose query applies the model. You load the registered model from Unity Catalog as a Spark UDF and call it inside the pipeline's final table definition — so every new record is scored as it flows through.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>import dlt
import mlflow
from pyspark.sql.functions import struct
</br>
<strong># Load the registered model from Unity Catalog as a Spark UDF</strong>
mlflow.set_registry_uri("databricks-uc")
predict_udf = mlflow.pyfunc.spark_udf(
    spark, "models:/main.ml_models.churn_model@champion"
)
</br>
<strong># Final pipeline step: score each record as it streams in</strong>
@dlt.table(name="churn_predictions")
def churn_predictions():
    features = dlt.read_stream("clean_data")
    return features.withColumn("prediction", predict_udf(struct("*")))
</code>
</pre>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:14pt; line-height:1.55;">
    The streaming inference pattern lets you use the same <code>pyfunc</code> model for both batch and streaming pipelines. By applying the model UDF in the final step, every new record is scored automatically as it arrives, enabling real-time predictions and seamless integration with upstream data cleaning and transformation steps.
  </div>
</div>

## Conclusion

In this lecture, you covered pipeline (streaming) deployment on Databricks:

1. **Pipeline / Streaming Deployment Fundamentals** — Streaming deployment continuously performs inference as data arrives, providing lower latency than batch and enabling event-driven systems to act sooner. It is more costly, more complex, and lower-throughput than batch, and it is not meant for true sub-second real-time predictions.
2. **Lakeflow Spark Declarative Pipelines** — SDP is the declarative way to do ETL and inference: you declare tables in SQL or Python and SDP orchestrates the DAG, manages infrastructure, and unifies batch and streaming. Its building blocks are streaming tables and materialized views — datasets defined by a query and kept up to date by a pipeline.
3. **Pipeline Features** — Expectations enforce data quality with warn/drop/fail policies; lineage, logging, monitoring, and notifications provide observability; and Auto Loader ingests new files incrementally with schema inference and evolution.
4. **Building and Running a Pipeline** — Write table definitions, create a pipeline from them, and click start. Making the final step a table whose query applies a `pyfunc` model UDF turns the pipeline into a streaming inference pipeline.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>